# Importing Libraries

In [1]:
from umap import UMAP # for UMAP latent space projections
import sys # for relatove imports of sigma

Using relative imports for sigma and its packages whilst they are being redeveloped

In [2]:
sys.path.insert(0,"..")
from sigma.utils.base import *
from sigma.utils import normalisation as norm 
from sigma.utils import visualisation as visual

from sigma.src.utils import same_seeds
from sigma.src.dim_reduction import Experiment
from sigma.models.autoencoder import AutoEncoder
from sigma.src.segmentation import PixelSegmenter
from sigma.gui import gui
from sigma.utils.loadtem import TEMDataset
from sigma.utils.load import *

## Loading the data from a `.rpl` file

Details on how to export the required files from aztec can be found in the `guides` folder


In [3]:
file_path='ryugu_aztec_data/EDS444.rpl'
bse_img_path='ryugu_aztec_data/BSE444.tif' #pointing to the bse image used for navigation / overlays
#file_path='23004_188/site 7/EDS Data.rpl'
sem=SEMDataset(file_path, nag_file_path=bse_img_path)

reading parameters from ryugu_aztec_data/EDS444.par
Unable to read X-Ray lines from sample metadata, setting blank list


In [4]:
sem.feature_list=['Fe_Ka', 'Mg_Ka', 'O_Ka', 'S_Ka', 'Si_Ka', 'P_Ka', 'Ni_Ka']

# Calibration of the energy axis

It is neccesary to apply a linear offset correction to the spectral axis of the form:
$E_{corrected}​=a⋅E_{measured}​+b$

This is done automatically with the `calibrate_spectra` dataset.

It will attempt to calibrate automatically, but when there are many peaks in a similar region, it is more robust to provide a dictionary of measured values. These measured peak energies can be found by inspecting the sum spectra with `gui.view_dataset`

In [5]:
gui.view_dataset(sem)

Output()

Output()

In [6]:
measured_peaks={'O_Ka':0.703,'Fe_Ka':6.44}

In [7]:
sem.calibrate_spectra(measured_peaks_dict=measured_peaks)

[Manual] Using O_Ka: Measured=0.703 keV → Reference=0.525 keV
[Manual] Using Fe_Ka: Measured=6.440 keV → Reference=6.404 keV
Calibration correction: E_corrected = 1.024752 * E_measured + -0.195400
Calibration successful.


In [8]:
sem.rebin_signal(size=(3,3)) # rebinning to improve quality of elemental maps

Rebinning the intensity with the size of (3, 3)


(<EDSSEMSpectrum, title: , dimensions: (170, 117|2048)>,
 <Signal2D, title: , dimensions: (|170, 117)>)

In [9]:
gui.view_dataset(sem)

Output()

Output()

# Processing the dataset ahead of projection and clustering

In [27]:
sem.normalisation([norm.neighbour_averaging,norm.zscore])

Set feature_list to ['Al_Ka', 'Ca_Ka', 'Fe_Ka', 'Mg_Ka', 'Mn_Ka', 'Na_Ka', 'Ni_La', 'O_Ka', 'S_Ka', 'Si_Ka', 'Ti_Ka']
Normalise dataset using:
    1. neighbour_averaging
    2. zscore


In [28]:
gui.view_pixel_distributions(sem,norm_list=[norm.neighbour_averaging,norm.zscore,norm.softmax],cmap='Reds')

# Projection with UMAP

In [29]:
data = sem.normalised_elemental_data.reshape(-1,len(sem.feature_list))
umap = UMAP(
        n_neighbors=20,
        min_dist=0.05,
        n_components=2,
        metric='euclidean'
    )
latent = umap.fit_transform(data)

# Clustering with GMM

In [14]:
ps_gmm=PixelSegmenter(latent=latent,dataset=sem,method='GaussianMixture',method_args={'n_components' :50, 'random_state':6, 'init_params':'kmeans'} )

## Clustering with HDBSCAN

In [30]:
ps_hdb = PixelSegmenter(latent=latent, 
                    dataset=sem,
                    method="HDBSCAN",
                    method_args=dict(min_cluster_size=10, min_samples=10,
                                     max_cluster_size=int(len(latent)/10),
                                     cluster_selection_epsilon=1e-1) )

# Plotting the clusters interactively

In [31]:
gui.interactive_latent_plot(ps=ps_hdb,ratio_to_be_shown=1.0)

In [ ]:
gui.check_latent_space(ps=ps_hdb,show_map=True,ratio_to_be_shown=1.0)

In [33]:
gui.view_clusters_sum_spectra(ps=ps_hdb)

SelectMultiple(options=('cluster_0', 'cluster_1', 'cluster_2', 'cluster_3', 'cluster_4', 'cluster_5', 'cluster…

Output()

In [34]:
gui.view_phase_map(ps=ps_hdb)

In [35]:
gui.show_cluster_distribution(ps=ps_hdb)

Output()

# NMF

In [36]:
weights, components = ps_hdb.get_unmixed_spectra_profile(clusters_to_be_calculated='All', 
                                                 n_components=3,
                                                 normalised=False, 
                                                 method='NMF', 
                                                 method_args={'init':'nndsvd'})

In [37]:
gui.show_unmixed_weights_and_compoments(ps=ps_hdb, weights=weights, components=components)

In [38]:
gui.show_abundance_map(ps=ps_hdb, weights=weights, components=components)

Output()